In [ ]:
import os
if "COLAB_" in "".join(os.environ.keys()):
    print("Running in Google Colab environment.")
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_dataset, Dataset

/home/mnikiema/OpenSource/mt-training/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


- Load NLLB model

In [2]:
model_name = "facebook/nllb-200-distilled-600M"

model = AutoModelForSeq2SeqLM.from_pretrained(
                                            model_name,
                                            device_map='cpu',
                                            use_cache=False
                                            )

Loading weights: 100%|██████████| 512/512 [00:00<00:00, 44377.75it/s]


In [3]:
french= "Tu es mon semblable."
moore= "Foo la mam bogre."
english= "You are my fellow."

In [16]:
src_lang = "fra_Latn"
tgt_lang = "mos_Latn"
tokenizer = AutoTokenizer.from_pretrained(model_name, src_lang=src_lang, tgt_lang=tgt_lang)

In [31]:
simple_example = tokenizer(french, src_lang=src_lang,
tgt_lang=tgt_lang, text_target=moore)

In [32]:
print(simple_example)

{'input_ids': [256057, 2374, 335, 1724, 65609, 2300, 248075, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1], 'labels': [256123, 277, 10770, 82, 2061, 10700, 119, 248075, 2]}


In [34]:
text = tokenizer.decode(simple_example["input_ids"], skip_special_tokens=False)
print(text)
labels = tokenizer.decode(simple_example["labels"], skip_special_tokens=False)
print(labels)

fra_Latn Tu es mon semblable.</s>
mos_Latn Foo la mam bogre.</s>


In [ ]:
inputs = tokenizer(french, src_lang="fra_Latn", return_tensors="pt")
outputs = model.generate(
    **inputs,
    forced_bos_token_id=tokenizer.convert_tokens_to_ids("mos_Latn")
)
translation = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(translation)

Yãmb yaa wa maam.


In [ ]:
def tokenize_fn(examples):
    inputs = examples["source"]
    targets = examples["target"]
    src_langs = examples["src_lang"]
    tgt_langs = examples["tgt_lang"]

    input_ids = []
    labels = []

    for src, tgt, src_lang, tgt_lang in zip(inputs, targets, src_langs, tgt_langs):
        tokenized = tokenizer(
            src,
            text_target=tgt,
            src_lang=src_lang,
            tgt_lang=tgt_lang,
            max_length=512,
            padding="max_length",
            truncation=True,
        )
        input_ids.append(tokenized["input_ids"])
        labels.append(tokenized["labels"])
    return {"input_ids": input_ids, "labels": labels}

In [ ]:
import evaluate
import numpy as np

bleu_metric = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = [[l] for l in tokenizer.batch_decode(labels, skip_special_tokens=True)]

    return {
        "bleu": bleu_metric.compute(predictions=decoded_preds, references=decoded_labels)["score"],
        "chrf": chrf_metric.compute(predictions=decoded_preds, references=decoded_labels)["score"],
    }

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
from huggingface_hub import HfFolder
import os

epochs = 2
learning_rate = 5e-5
batch_size = 4

hf_id = "madoss"
output_dir = os.path.join(hf_id, "nllb-200-finetuned-600-FRA-MOS")

# Define training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    num_train_epochs=epochs,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    gradient_accumulation_steps=4,
    eval_accumulation_steps=4,
    #gradient_checkpointing=True,

    fp16=True,
    fp16_full_eval=True,

    learning_rate=learning_rate,
    lr_scheduler_type='constant',  # "constant", "linear", "cosine"

    eval_strategy="steps",  # or "epoch"
    eval_steps=100,
    save_strategy="epoch",
    predict_with_generate=True,
    generation_max_length=128,
    logging_steps=100,
    report_to="trackio", # "tensorboard", "wandb", or "none"

    # push to hub parameters
    push_to_hub=True,
    hub_private_repo=True,
    hub_strategy="end",  # "every_save" or "end"
    hub_model_id=output_dir,
    hub_token=HfFolder.get_token(),
)

In [ ]:
# Initialize the trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=...,
    eval_dataset=...,
    compute_metrics=compute_metrics,
)